# 13: Union-rule dependence (partialling out marginals) and ensemble size (EXPLORATORY)

## Part A: does severity-weighted dependence predict union-rule benefit?

Notebook 12 tested dependence against the **matched-workload** outcome, where truncation breaks
the relationship. Under the **union** rule it is close to an identity:

    weighted advantage = (better model's weighted miss cost) - (weighted cost of shared misses)

and shared misses are exactly what dependence measures. Two things were therefore missing:
the union outcome, and **partialling out the marginals** (pairs differ in model quality, which
swamps the dependence term in a raw correlation).

Run in three steps:
1. **Identity check, same dataset.** Regress fresh union advantage on fresh marginals and fresh
   dependence. Should be near-deterministic; if not, the accounting is wrong.
2. **Variance split.** How much of the spread in union advantage across the 55 pairs comes from
   differences in marginal quality versus differences in dependence.
3. **Out of sample and useful.** Does *validation*-measured dependence predict *fresh* union
   advantage after controlling for validation marginals? This is the one that matters for
   choosing a partner in advance, and it is not guaranteed: dependence is estimated from a few
   dozen severe misses and may be too noisy to transfer.

## Part B: does ensemble size help?

v1 only ever tested pairs (55 = C(11,2)); the 2x2 factorial varied the aggregation rule, not the
number of models. Here k = 2 to 5 (matched seeds), under both rules, with referral cost.

Fresh set, 25% burden, weights 500/200/15, false positive 1. CPU, about 20 minutes.

In [1]:
# Cell 1: load
import io, os, json, itertools, datetime, time
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
svc=gbuild('drive','v3')
F_PRED='1nCloJ5o9veZb2xJL_2EjdVHQXN31jy4w'; F_RES='1-HT4ONiwIfuBtC19s7QkDd8Euklh0Xon'; F_FIG='18eyHRWuqFcUrYiL8nPtpIalV7CreLkMK'
def dl(fid,p):
    with io.FileIO(p,'wb') as fh:
        d=MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid)); done=False
        while not done: _,done=d.next_chunk()
def find_id(n,f):
    r=svc.files().list(q=f"'{f}' in parents and name='{n}' and trashed=false", fields='files(id)').execute().get('files',[])
    return r[0]['id'] if r else None
def up(p,f,m):
    svc.files().create(body={'name':os.path.basename(p),'parents':[f]}, media_body=MediaFileUpload(p,mimetype=m), fields='id').execute()
    print('  uploaded', os.path.basename(p))

ARCHS=['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
       'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
       'retfound_binary','retfound_5class','retfound_adversarial']
SEEDS=[0,1,2]; PAIRS=list(itertools.combinations(ARCHS,2)); COMBOS=[(i,j) for i in SEEDS for j in SEEDS]
def tag(k): return '' if k==0 else f'_seed{k}'
BURDEN=0.25; W={4:500,3:200,2:15}; WFP=1
def load(prefix, split):
    P, ref = {}, None
    for a in ARCHS:
        for k in SEEDS:
            fn=f'{prefix}{a}{tag(k)}_v2.csv'
            if not os.path.exists(fn): dl(find_id(fn,F_PRED), fn)
            d=pd.read_csv(fn); assert set(d['split'])=={split}
            if ref is None: ref=d[['image_id','true_grade','true_label']].reset_index(drop=True)
            P[(a,k)]=d['predicted_probability'].to_numpy(np.float64)
    return P, ref
PV,RV=load('eyepacs_predictions_val_','val'); PF,RF=load('eyepacs_predictions_fresh_','fresh')
def setup(R_):
    G=R_['true_grade'].to_numpy(int); L=R_['true_label'].to_numpy(int); n=len(G)
    missw=np.where(L==1, np.where(G==4,W[4],np.where(G==3,W[3],W[2])), 0.0)
    return G,L,n,int(round(BURDEN*n)),missw
GV,LV,NV,RV_,MWV=setup(RV); GF,LF,NF,RF_,MWF=setup(RF)
print(f'validation {NV} (R={RV_}) | fresh {NF} (R={RF_})')

validation 2900 (R=725) | fresh 15794 (R=3948)


In [2]:
# Cell 2: union-rule quantities, severity weighted
def topR(p,R):
    m=np.zeros(len(p),bool); m[np.argsort(-p,kind='stable')[:R]]=True; return m

def union_quantities(P,G,L,n,R,missw):
    REF={k:topR(v,R) for k,v in P.items()}
    grades=[2,3,4]; ng={g:int((G==g).sum()) for g in grades}
    misscost=lambda m: float(missw[(~m)&(L==1)].sum())
    rows=[]
    for a,b in PAIRS:
        acc={'marg_min':[], 'shared_cost':[], 'shared_indep':[], 'dep_excess':[],
             'adv_union_miss_only':[], 'adv_union_same_volume':[], 'pdr_union_avoided':[]}
        for i,j in COMBOS:
            ra,rb=REF[(a,i)],REF[(b,j)]
            ca,cb=misscost(ra),misscost(rb)
            u=ra|rb; R_or=int(u.sum())
            shared=float(missw[(~ra)&(~rb)&(L==1)].sum())
            # independence benchmark, computed per grade so severity is respected
            si=0.0
            for g in grades:
                mg=(G==g)
                pa=((~ra)&mg).sum()/max(1,ng[g]); pb=((~rb)&mg).sum()/max(1,ng[g])
                si += W[g]*ng[g]*pa*pb
            base_p = P[(a,i)] if ca<=cb else P[(b,j)]
            acc['marg_min'].append(min(ca,cb))
            acc['shared_cost'].append(shared); acc['shared_indep'].append(si)
            acc['dep_excess'].append(shared-si)
            acc['adv_union_miss_only'].append(min(ca,cb)-shared)
            acc['adv_union_same_volume'].append(misscost(topR(base_p,R_or))-shared)
            g4=(G==4)
            acc['pdr_union_avoided'].append(float(min(((~ra)&g4).sum(),((~rb)&g4).sum()) - ((~ra)&(~rb)&g4).sum()))
        rows.append({'model_a':a,'model_b':b, **{k:float(np.mean(v)) for k,v in acc.items()}})
    return pd.DataFrame(rows).set_index(['model_a','model_b'])

t0=time.time(); UV=union_quantities(PV,GV,LV,NV,RV_,MWV); UF=union_quantities(PF,GF,LF,NF,RF_,MWF)
print(f'{time.time()-t0:.0f}s')
print(UF[['marg_min','shared_cost','shared_indep','dep_excess','adv_union_miss_only','pdr_union_avoided']].describe().round(1).to_string())

1s
       marg_min  shared_cost  shared_indep  dep_excess  adv_union_miss_only  pdr_union_avoided
count      55.0         55.0          55.0        55.0                 55.0               55.0
mean    23657.5      14850.1        5094.5      9755.7               8807.3                6.9
std      2599.7       1922.7         521.1      1739.6               2134.6                2.6
min     20680.0      12327.2        4210.7      7885.7               2632.2                1.2
25%     21501.7      13335.3        4785.5      8412.1               7786.9                4.9
50%     23934.4      14222.8        5067.7      9055.9               8758.9                6.6
75%     25494.4      16102.2        5347.5     10670.3              10285.3                9.1
max     31235.6      20971.7        6335.3     14636.4              12036.7               12.0


In [3]:
# Cell 3: identity check, variance split, and the out-of-sample partial test
idx=[(a,b) for a,b in PAIRS]
def col(df,c): return np.array([df.loc[p,c] for p in idx])
def z(x): return (x-x.mean())/x.std(ddof=0)
def ols(y, X):
    Xd=np.column_stack([np.ones(len(y))]+[z(c) for c in X]); yz=z(y)
    beta,*_=np.linalg.lstsq(Xd,yz,rcond=None)
    pred=Xd@beta; r2=1-((yz-pred)**2).sum()/((yz-yz.mean())**2).sum()
    return beta[1:], r2
def partial_spearman(y,x,ctrl):
    ry=y-np.polyval(np.polyfit(ctrl,y,1),ctrl); rx=x-np.polyval(np.polyfit(ctrl,x,1),ctrl)
    return float(spearmanr(rx,ry).correlation)

# 1. identity check on the same dataset
y_f=col(UF,'adv_union_miss_only'); marg_f=col(UF,'marg_min'); dep_f=col(UF,'dep_excess')
b_same,r2_same=ols(y_f,[marg_f,dep_f])
# 2. variance split: contribution of each standardised term
var_marg=float(np.var(marg_f)); var_dep=float(np.var(col(UF,'shared_cost')))
# 3. out of sample: validation predictors -> fresh outcome
marg_v=col(UV,'marg_min'); dep_v=col(UV,'dep_excess'); shared_v=col(UV,'shared_cost')
b_oos,r2_oos=ols(y_f,[marg_v,dep_v])
rng=np.random.default_rng(20260922)
def mlci(fn,B=2000):
    out=[]
    for _ in range(B):
        s=rng.choice(ARCHS,len(ARCHS),replace=True)
        ps=[(u,v) if (u,v) in UF.index else (v,u) for u,v in itertools.combinations(s,2) if u!=v]
        if len(set(ps))<6: continue
        k=[idx.index(p) for p in ps]
        try: out.append(fn(k))
        except Exception: pass
    return [round(float(v),3) for v in np.nanpercentile(out,[2.5,97.5])]

ps_same=partial_spearman(y_f,dep_f,marg_f)
ps_oos =partial_spearman(y_f,dep_v,marg_v)
ci_oos = mlci(lambda k: partial_spearman(y_f[k],dep_v[k],marg_v[k]))
raw_oos=float(spearmanr(dep_v,y_f).correlation)

PARTA={
 'identity_same_dataset':{'std_beta_marginal':round(float(b_same[0]),3),'std_beta_dependence':round(float(b_same[1]),3),
                          'r2':round(float(r2_same),4),
                          'note':'advantage = better marginal miss cost minus shared miss cost, so this should be near 1'},
 'spread_across_pairs':{'sd_marginal_miss_cost':round(float(np.sqrt(var_marg)),1),
                        'sd_shared_miss_cost':round(float(np.sqrt(var_dep)),1)},
 'raw_out_of_sample_spearman_dependence_vs_advantage':round(raw_oos,3),
 'partial_same_dataset_controlling_marginals':round(ps_same,3),
 'partial_out_of_sample_controlling_marginals':round(ps_oos,3),
 'partial_out_of_sample_model_level_ci':ci_oos,
 'oos_regression':{'std_beta_marginal':round(float(b_oos[0]),3),'std_beta_dependence':round(float(b_oos[1]),3),
                   'r2':round(float(r2_oos),4)},
}
print(json.dumps(PARTA, indent=2))
print('\nNegative dependence coefficients mean: less shared failure, more union benefit (the hypothesis).')

{
  "identity_same_dataset": {
    "std_beta_marginal": 1.017,
    "std_beta_dependence": -0.784,
    "r2": 0.9767,
    "note": "advantage = better marginal miss cost minus shared miss cost, so this should be near 1"
  },
  "spread_across_pairs": {
    "sd_marginal_miss_cost": 2576.0,
    "sd_shared_miss_cost": 1905.1
  },
  "raw_out_of_sample_spearman_dependence_vs_advantage": 0.166,
  "partial_same_dataset_controlling_marginals": -0.944,
  "partial_out_of_sample_controlling_marginals": 0.009,
  "partial_out_of_sample_model_level_ci": [
    -0.636,
    0.415
  ],
  "oos_regression": {
    "std_beta_marginal": 0.561,
    "std_beta_dependence": -0.365,
    "r2": 0.327
  }
}

Negative dependence coefficients mean: less shared failure, more union benefit (the hypothesis).


In [4]:
# Cell 4: Part B, ensemble size k = 2 to 5 (matched seeds)
REF_F={k:topR(v,RF_) for k,v in PF.items()}; RK_F={k:rankdata(v)/NF for k,v in PF.items()}
misscost=lambda m: float(MWF[(~m)&(LF==1)].sum())
g4=(GF==4)
rows=[]
rng2=np.random.default_rng(7)
for k in (2,3,4,5):
    combos=list(itertools.combinations(ARCHS,k))
    if len(combos)>350: combos=[combos[i] for i in rng2.choice(len(combos),350,replace=False)]
    for grp in combos:
        for s in SEEDS:                                   # matched seeds
            refs=[REF_F[(m,s)] for m in grp]; rks=[RK_F[(m,s)] for m in grp]
            costs=[misscost(r) for r in refs]; best=int(np.argmin(costs))
            u=np.logical_or.reduce(refs); R_or=int(u.sum())
            mx=np.maximum.reduce(rks); matched=topR(mx,RF_)
            base_same_vol=topR(PF[(grp[best],s)],R_or)
            rows.append({'k':k,'seed':s,
                         'adv_matched':costs[best]-misscost(matched),
                         'adv_union_same_volume':misscost(base_same_vol)-misscost(u),
                         'adv_union_miss_only':costs[best]-misscost(u),
                         'extra_referrals':R_or-RF_,
                         'pdr_matched':float(((~refs[best])&g4).sum()-((~matched)&g4).sum()),
                         'pdr_union':float(((~refs[best])&g4).sum()-((~u)&g4).sum())})
K=pd.DataFrame(rows)
KS=K.groupby('k').agg(['mean']).round(1)
print(K.groupby('k')[['adv_matched','adv_union_same_volume','adv_union_miss_only','extra_referrals','pdr_matched','pdr_union']]
      .mean().round(2).to_string())
print('\nadv_matched: fixed referral volume. adv_union_same_volume: union vs best member given the')
print('same enlarged volume. extra_referrals: what the union costs, out of', NF, 'images.')

   adv_matched  adv_union_same_volume  adv_union_miss_only  extra_referrals  pdr_matched  pdr_union
k                                                                                                  
2      2566.76                2081.24              8738.48          1196.45         3.19       6.99
3      3104.20                2639.62             11494.71          1947.36         3.55       8.08
4      3068.62                2741.21             12849.17          2489.56         2.92       8.32
5      2934.17                2928.31             13656.34          2906.39         2.34       8.40

adv_matched: fixed referral volume. adv_union_same_volume: union vs best member given the
same enlarged volume. extra_referrals: what the union costs, out of 15794 images.


In [5]:
# Cell 5: figures and save
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,3,figsize=(15,4.4))
ax[0].scatter(dep_f, y_f, s=18, c='C0'); ax[0].set_xlabel('fresh dependence excess (weighted)')
ax[0].set_ylabel('fresh union advantage'); ax[0].set_title('same dataset (identity)')
ax[1].scatter(dep_v, y_f, s=18, c='C1'); ax[1].set_xlabel('validation dependence excess (weighted)')
ax[1].set_ylabel('fresh union advantage'); ax[1].set_title('out of sample')
m=K.groupby('k').mean(numeric_only=True)
ax[2].plot(m.index, m['adv_matched'], 'o-', label='matched volume')
ax[2].plot(m.index, m['adv_union_same_volume'], 's-', label='union vs same volume')
ax[2].set_xlabel('number of models (k)'); ax[2].set_ylabel('weighted advantage'); ax[2].legend()
ax[2].set_title('does ensemble size help?'); ax[2].axhline(0,color='grey',lw=0.8)
fig.suptitle('Union-rule dependence and ensemble size (exploratory, fresh set)')
fig.tight_layout(); fig.savefig('union_dependence_and_k_v3.png', dpi=200)

OUT={'generated_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
     'status':'EXPLORATORY','burden':BURDEN,'weights':W,
     'part_a_union_dependence':PARTA,
     'part_b_ensemble_size':K.groupby('k').mean(numeric_only=True).round(3).to_dict(orient='index')}
with open('union_dependence_and_k_v3.json','w') as f: json.dump(OUT,f,indent=2,default=str)
K.to_csv('ensemble_size_detail_v3.csv',index=False)
UF.reset_index().to_csv('union_quantities_fresh_v3.csv',index=False)
UV.reset_index().to_csv('union_quantities_val_v3.csv',index=False)
for fn,mt in (('union_dependence_and_k_v3.json','application/json'),('ensemble_size_detail_v3.csv','text/csv'),
              ('union_quantities_fresh_v3.csv','text/csv'),('union_quantities_val_v3.csv','text/csv')):
    up(fn,F_RES,mt)
up('union_dependence_and_k_v3.png',F_FIG,'image/png')
print('saved')

  uploaded union_dependence_and_k_v3.json
  uploaded ensemble_size_detail_v3.csv
  uploaded union_quantities_fresh_v3.csv
  uploaded union_quantities_val_v3.csv
  uploaded union_dependence_and_k_v3.png
saved
